# IBVAP — Clean Colab Notebook

**Intelligent Border Video Analytics Pipeline**

This notebook reorganizes the supplied IBVAP workflow into logical stages:

- Environment & project setup
- ExDark dataset preparation
- YOLOv8 detector training and validation
- Tracking and security analytics
- ANPR + EasyOCR
- Face privacy redaction
- CCTV inference and benchmarking
- Multi-camera geometry
- License-plate training
- Multi-terrain dataset expansion
- Motion-gating optimization
- Showcase and metrics

> **Credential safety:** the Roboflow API key from the source file is not included here. Set `ROBOFLOW_API_KEY` in the Colab environment before running Roboflow cells.

## 1. Environment & project setup

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

!pip install -q ultralytics supervision easyocr opencv-python-headless

from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import torch
import json
import os

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/IBVAP"
folders = ["datasets", "models", "videos", "outputs", "metrics", "showcase"]

for folder in folders:
    os.makedirs(f"{BASE}/{folder}", exist_ok=True)

print("IBVAP directory structure initialized.")

import os

BASE = "/content/drive/MyDrive/IBVAP"

print("BASE:", BASE)
print("Exists:", os.path.exists(BASE))

exdark_dir = f"{BASE}/datasets/exdark"

print("ExDark directory:", exdark_dir)
print("Exists:", os.path.exists(exdark_dir))

if os.path.exists(exdark_dir):
    print("\nContents:")
    for item in os.listdir(exdark_dir):
        print(" -", item)

import os

BASE = "/content/drive/MyDrive/IBVAP"

print("IBVAP exists:", os.path.exists(BASE))

if os.path.exists(BASE):
    print("\nIBVAP contents:")
    for item in os.listdir(BASE):
        print(" -", item)

datasets_dir = f"{BASE}/datasets"

print("Datasets exists:", os.path.exists(datasets_dir))

if os.path.exists(datasets_dir):
    print("\nDatasets contents:")
    for item in os.listdir(datasets_dir):
        print(" -", item)

import os

EXDARK_DIR = "/content/drive/MyDrive/IBVAP/datasets/exdark"

os.makedirs(EXDARK_DIR, exist_ok=True)

print("Created:", EXDARK_DIR)

## 2. ExDark download, inspection & initial training

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("chs-s7apx").project("exdark-aiswr")
version = project.version(1)
dataset = version.download("yolov8")

import os

EXDARK_DIR = "/content/drive/MyDrive/IBVAP/datasets/exdark"

print("ExDark exists:", os.path.exists(EXDARK_DIR))

if os.path.exists(EXDARK_DIR):
    print("\nContents:")
    for item in os.listdir(EXDARK_DIR):
        print(" -", item)

for root, dirs, files in os.walk(EXDARK_DIR):
    level = root.replace(EXDARK_DIR, "").count(os.sep)

    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

yaml_path = f"{EXDARK_DIR}/data.yaml"

print("data.yaml exists:", os.path.exists(yaml_path))

if os.path.exists(yaml_path):
    print("\n========== data.yaml ==========\n")
    with open(yaml_path, "r") as f:
        print(f.read())

import os

print(os.path.exists("/content/exdark-1"))

if os.path.exists("/content/exdark-1"):
    print(os.listdir("/content/exdark-1"))

import os
import shutil

BASE = "/content/drive/MyDrive/IBVAP"
SOURCE = "/content/exdark-1"
DEST = f"{BASE}/datasets/exdark"

print("Source exists:", os.path.exists(SOURCE))
print("Destination exists:", os.path.exists(DEST))

# Copy dataset into the empty Drive folder
if os.path.exists(SOURCE) and os.path.exists(DEST):
    shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)
    print("\n✅ Dataset copied successfully!")
else:
    print("❌ Check the source/destination paths.")

for root, dirs, files in os.walk(DEST):
    level = root.replace(DEST, "").count(os.sep)

    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

yaml_path = f"{DEST}/data.yaml"

print("========== data.yaml ==========\n")

with open(yaml_path, "r") as f:
    print(f.read())

from collections import Counter

image_dir = f"{BASE}/datasets/exdark/train/images"
label_dir = f"{BASE}/datasets/exdark/train/labels"

print(f"Training images: {len(os.listdir(image_dir))}")
print(f"Training labels: {len(os.listdir(label_dir))}")

counter = Counter()
for file in os.listdir(label_dir):
    if file.endswith(".txt"):
        with open(os.path.join(label_dir, file)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1

print("Class distribution:", counter)

# Cell 6: Train on ExDark
model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{BASE}/datasets/exdark/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    workers=2,
    project=f"{BASE}/outputs",
    name="exdark_yolov8n"
)

# Cell 7: Backup best model
!cp {BASE}/outputs/exdark_yolov8n/weights/best.pt {BASE}/models/ibvap_exdark_v1.pt
print("ExDark v1 model saved to Drive.")

# Cell 8: Evaluate ExDark model
model = YOLO(f"{BASE}/models/ibvap_exdark_v1.pt")

metrics = model.val(
    data=f"{BASE}/datasets/exdark/data.yaml",
    imgsz=640
)

# Extract core metrics for your presentation slides
print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

# Save metrics to text
with open(f"{BASE}/metrics/exdark_v1.txt", "w") as f:
    f.write(f"mAP50: {metrics.box.map50:.3f}\nPrecision: {metrics.box.mp:.3f}\nRecall: {metrics.box.mr:.3f}")

## 3. Core analytics, ANPR & privacy utilities

In [ ]:
import numpy as np
from collections import deque

# Load your fine-tuned ExDark model
detector = YOLO(f"{BASE}/models/ibvap_detector.pt")

def ccw(A, B, C):
    return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])

def intersect(A, B, C, D):
    return ccw(A, C, D) != ccw(B, C, D) and ccw(A, B, C) != ccw(A, B, D)

class VirtualFence:
    def __init__(self, line_p1, line_p2):
        self.line_p1 = line_p1
        self.line_p2 = line_p2
        self.track_history = {}
        self.breaches = set()

    def update(self, track_id, bbox):
        x1, y1, x2, y2 = bbox
        bottom_center = ((x1 + x2) / 2, y2)
        if track_id not in self.track_history:
            self.track_history[track_id] = []
        self.track_history[track_id].append(bottom_center)

        if len(self.track_history[track_id]) >= 2:
            prev_pt = self.track_history[track_id][-2]
            curr_pt = self.track_history[track_id][-1]
            if intersect(prev_pt, curr_pt, self.line_p1, self.line_p2):
                if track_id not in self.breaches:
                    self.breaches.add(track_id)
                    return True
        return False

class RestrictedZone:
    def __init__(self, polygon, dwell_threshold_frames=45, speed_threshold_px=35.0):
        self.polygon = np.array(polygon, np.int32)
        self.dwell_threshold = dwell_threshold_frames
        self.speed_threshold = speed_threshold_px
        self.entry_frames = {}
        self.last_positions = {}
        self.alerted_loitering = set()

    def update(self, track_id, bbox, frame_idx):
        x1, y1, x2, y2 = bbox
        cx, cy = int((x1 + x2) / 2), int(y2)
        is_inside = cv2.pointPolygonTest(self.polygon, (cx, cy), False) >= 0
        alerts = []

        if is_inside:
            if track_id not in self.entry_frames:
                self.entry_frames[track_id] = frame_idx
            dwell = frame_idx - self.entry_frames[track_id]
            if dwell >= self.dwell_threshold and track_id not in self.alerted_loitering:
                self.alerted_loitering.add(track_id)
                alerts.append({"type": "LOITERING_ALERT", "track_id": int(track_id), "dwell_frames": int(dwell)})
        else:
            self.entry_frames.pop(track_id, None)
            self.alerted_loitering.discard(track_id)

        if track_id in self.last_positions:
            prev_cx, prev_cy = self.last_positions[track_id]
            speed = np.sqrt((cx - prev_cx)**2 + (cy - prev_cy)**2)
            if speed > self.speed_threshold:
                alerts.append({"type": "SUDDEN_MOVEMENT", "track_id": int(track_id), "speed_px": round(float(speed), 2)})

        self.last_positions[track_id] = (cx, cy)
        return alerts

class VehicleTracker:
    def __init__(self, history_len=30):
        self.trajectories = {}
        self.history_len = history_len
        self.wrong_way_alerts = set()

    def update_trajectory(self, track_id, bbox, frame_idx):
        x1, y1, x2, y2 = bbox
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        if track_id not in self.trajectories:
            self.trajectories[track_id] = deque(maxlen=self.history_len)
        self.trajectories[track_id].append((cx, cy))

        direction = "STATIONARY"
        events = []
        if len(self.trajectories[track_id]) >= 10:
            dx = self.trajectories[track_id][-1][0] - self.trajectories[track_id][0][0]
            dy = self.trajectories[track_id][-1][1] - self.trajectories[track_id][0][1]
            direction = ("RIGHT" if dx > 0 else "LEFT") if abs(dx) > abs(dy) else ("DOWN" if dy > 0 else "UP")

            if direction == "DOWN" and track_id not in self.wrong_way_alerts:
                self.wrong_way_alerts.add(track_id)
                events.append({"frame": frame_idx, "type": "WRONG_WAY_VEHICLE", "track_id": int(track_id), "direction": direction})

        return direction, events

import easyocr
import re

# Load OCR engine
reader = easyocr.Reader(['en'], gpu=True)

# Load / Download License Plate Model
plate_model_path = f"{BASE}/models/license_plate_detector.pt"
if not os.path.exists(plate_model_path):
    !wget -q -O {plate_model_path} "https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/best.pt"
plate_detector = YOLO(plate_model_path)

# Load / Download Face Model
face_model_path = f"{BASE}/models/yolov8n-face.pt"
if not os.path.exists(face_model_path):
    !wget -q -O {face_model_path} "https://huggingface.co/Autsadin/yolov8-face/resolve/main/yolov8n-face.pt"
face_detector = YOLO(face_model_path)

def extract_plate_text(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    h, w = frame.shape[:2]
    cropped = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
    if cropped.size == 0:
        return None, 0.0

    gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    results = reader.readtext(gray)
    if not results:
        return None, 0.0

    texts, confs = [], []
    for (_, text, conf) in results:
        clean = re.sub(r'[^A-Z0-9]', '', text.upper())
        if len(clean) >= 4:
            texts.append(clean)
            confs.append(conf)
    return ("".join(texts), max(confs)) if texts else (None, 0.0)

def redact_faces(frame):
    results = face_detector(frame, conf=0.4, verbose=False)[0]
    if results.boxes:
        h, w = frame.shape[:2]
        for box in results.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            roi = frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)]
            if roi.size > 0:
                frame[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = cv2.GaussianBlur(roi, (51, 51), 30)
    return frame

def apply_clahe(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)

## 4. Base IBVAP inference pipeline & initial demonstrations

In [ ]:
import cv2
import json
import numpy as np
from datetime import datetime

# 1. LOAD THE NEW MODEL HERE
BASE = "/content/drive/MyDrive/IBVAP"
detector = YOLO(f"{BASE}/models/ibvap_detector.pt")

def run_ibvap(input_path, output_path, camera_id="BOP-01"):
    cap = cv2.VideoCapture(input_path)
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Initialize Analytics
    fence = VirtualFence(line_p1=(0, int(height * 0.70)), line_p2=(width, int(height * 0.70)))
    zone = RestrictedZone([(int(width*0.25), int(height*0.35)), (int(width*0.75), int(height*0.35)),
                           (int(width*0.75), int(height*0.85)), (int(width*0.25), int(height*0.85))])
    tracker = VehicleTracker()

    event_log = []
    frame_idx = 0
    plate_cache = set()
    vehicle_classes = {2, 3, 5, 7} # Car, motorcycle, bus, truck indices in YOLOv8n

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1

        # 1. Night Detection & Enhancement
        is_night = np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)) < 90
        if is_night: frame = apply_clahe(frame)

        # 2. Base Detection & Tracking (using your ExDark model)
        results = detector.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=0.35)[0]

        if results.boxes and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.int().cpu().numpy()
            classes = results.boxes.cls.int().cpu().numpy()

            for box, tid, cls_id in zip(boxes, track_ids, classes):
                cls_name = detector.names[cls_id]
                x1, y1, x2, y2 = map(int, box)
                color = (0, 255, 0)

                # Event Logging Template
                base_event = {"camera_id": camera_id, "timestamp": datetime.now().isoformat(),
                              "frame": frame_idx, "object_type": cls_name, "track_id": int(tid)}

                # Analytics: Perimeter Breach
                if fence.update(tid, box):
                    color = (0, 0, 255)
                    event_log.append({**base_event, "event_type": "PERIMETER_BREACH", "severity": "CRITICAL"})

                # Analytics: Restricted Zone
                for alert in zone.update(tid, box, frame_idx):
                    event_log.append({**base_event, "event_type": alert["type"], "severity": "WARNING"})

                # Analytics: Vehicles (ANPR & Wrong Way)
                if cls_id in vehicle_classes:
                    dir_status, v_events = tracker.update_trajectory(tid, box, frame_idx)
                    for ve in v_events:
                        event_log.append({**base_event, "event_type": ve["type"], "severity": "CRITICAL"})

                    plate_res = plate_detector(frame, conf=0.35, verbose=False)[0]
                    if plate_res.boxes:
                        for p_box in plate_res.boxes.xyxy.cpu().numpy():
                            text, conf = extract_plate_text(frame, p_box)
                            if text and text not in plate_cache:
                                plate_cache.add(text)
                                event_log.append({**base_event, "event_type": "ANPR", "plate": text, "confidence": conf, "severity": "INFO"})

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"ID:{tid} {cls_name}", (x1, max(20, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        # 3. Privacy Redaction
        frame = redact_faces(frame)

        # 4. Draw Geometry Overlays
        cv2.line(frame, fence.line_p1, fence.line_p2, (0, 0, 255), 2)
        cv2.polylines(frame, [zone.polygon], True, (255, 255, 0), 2)
        cv2.putText(frame, "NIGHT VISION ACTIVE" if is_night else "DAY MODE", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255) if is_night else (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()

    with open(f"{BASE}/outputs/alerts.json", "w") as f:
        json.dump(event_log, f, indent=2)
    print(f"Pipeline complete. Alerts saved to {BASE}/outputs/alerts.json")

# Download daytime test clip
!wget -q -O {BASE}/videos/day.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4

# Generate synthetic night clip
def make_night_version(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps,
                          (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        out.write(np.clip((frame.astype(np.float32) * 0.25) + np.random.normal(0, 8, frame.shape), 0, 255).astype(np.uint8))
    cap.release()
    out.release()

make_night_version(f"{BASE}/videos/day.mp4", f"{BASE}/videos/night_synthetic.mp4")
print("Test videos ready.")

import pandas as pd

print("Processing Day Video...")
run_ibvap(f"{BASE}/videos/day.mp4", f"{BASE}/showcase/day_result.mp4")

print("\nProcessing Night Video...")
run_ibvap(f"{BASE}/videos/night_synthetic.mp4", f"{BASE}/showcase/night_result.mp4")

print("\n====================================================")
print("              IBVAP DEMONSTRATION                   ")
print("====================================================")
print("✓ Environment: Auto-Night Detection & CLAHE")
print("✓ Detection: ExDark YOLOv8 (Humans & Vehicles)")
print("✓ Tracking: ByteTrack + Trajectories")
print("✓ Analytics: Virtual Fence, Loitering, Wrong-Way")
print("✓ ANPR: Plate Detection + EasyOCR")
print("✓ Privacy: Gaussian Face Redaction")
print("====================================================\n")

# Display the standardized event log
alerts_df = pd.read_json(f"{BASE}/outputs/alerts.json")
display(alerts_df.tail(10) if not alerts_df.empty else "No events logged.")

## 5. Unified master dataset & final detector training

In [ ]:
# Cell: One-time model loading (safe to re-run — skips reload if already loaded)

if 'detector' not in globals():
    detector = YOLO(f"{BASE}/models/ibvap_exdark_v1.pt")
    print("✓ Detector loaded")
else:
    print("✓ Detector already in memory — skipped reload")

if 'plate_detector' not in globals():
    plate_model_path = f"{BASE}/models/license_plate_detector.pt"
    if not os.path.exists(plate_model_path):
        get_ipython().system(f'wget -q -O {plate_model_path} "https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/best.pt"')
    plate_detector = YOLO(plate_model_path)
    print("✓ Plate detector loaded")
else:
    print("✓ Plate detector already in memory — skipped reload")

if 'face_detector' not in globals():
    face_model_path = f"{BASE}/models/yolov8n-face.pt"
    if not os.path.exists(face_model_path):
        get_ipython().system(f'wget -q -O {face_model_path} "https://huggingface.co/Autsadin/yolov8-face/resolve/main/yolov8n-face.pt"')
    face_detector = YOLO(face_model_path)
    print("✓ Face detector loaded")
else:
    print("✓ Face detector already in memory — skipped reload")

if 'reader' not in globals():
    reader = easyocr.Reader(['en'], gpu=True)
    print("✓ EasyOCR reader loaded")
else:
    print("✓ EasyOCR reader already in memory — skipped reload")

import os
import glob

# Master Mapping: ExDark Index -> New IBVAP Index
# ExDark: 10=People, 0=Bicycle, 4=Car, 9=Motorbike, 3=Bus
class_map = {10: 0, 0: 1, 4: 2, 9: 3, 3: 4}

def normalize_labels(label_dir):
    txt_files = glob.glob(os.path.join(label_dir, "*.txt"))
    removed_files = 0

    for txt_file in txt_files:
        valid_lines = []
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue

                old_cls = int(parts[0])
                if old_cls in class_map:
                    parts[0] = str(class_map[old_cls])
                    valid_lines.append(" ".join(parts))

        # Overwrite with only valid classes, or delete if empty (no relevant objects)
        if valid_lines:
            with open(txt_file, 'w') as f:
                f.write("\n".join(valid_lines) + "\n")
        else:
            os.remove(txt_file)
            image_file = txt_file.replace('/labels/', '/images/').replace('.txt', '.jpg')
            if os.path.exists(image_file): os.remove(image_file)
            removed_files += 1

    print(f"Processed {label_dir}. Removed {removed_files} empty frames.")

normalize_labels(f"{BASE}/datasets/exdark/train/labels")
normalize_labels(f"{BASE}/datasets/exdark/valid/labels")

import os
import shutil

def merge_dataset(source_name, split="train"):
    src_img = f"{BASE}/datasets/{source_name}/{split}/images"
    src_lbl = f"{BASE}/datasets/{source_name}/{split}/labels"

    dst_img = f"{BASE}/datasets/ibvap_master/{split}/images"
    dst_lbl = f"{BASE}/datasets/ibvap_master/{split}/labels"

    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    if not os.path.exists(src_img):
        return 0

    count = 0
    for file in os.listdir(src_img):
        # Prefix filenames to prevent naming collisions between datasets
        new_name = f"{source_name}_{file}"
        shutil.copy(os.path.join(src_img, file), os.path.join(dst_img, new_name))

        lbl_file = file.replace('.jpg', '.txt').replace('.png', '.txt')
        if os.path.exists(os.path.join(src_lbl, lbl_file)):
            new_lbl_name = new_name.replace('.jpg', '.txt').replace('.png', '.txt')
            shutil.copy(os.path.join(src_lbl, lbl_file), os.path.join(dst_lbl, new_lbl_name))
        count += 1
    return count

print("Merging ExDark into Master...")
train_count = merge_dataset("exdark", "train")
valid_count = merge_dataset("exdark", "valid")

print(f"Moved {train_count} training and {valid_count} validation frames.")

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("pinakpanighosh3-gmail-com").project("llvip-nhnrm")
version = project.version(4)
dataset = version.download("yolov8")

import os
import glob

def normalize_llvip(label_dir):
    if not os.path.exists(label_dir): return
    txt_files = glob.glob(os.path.join(label_dir, "*.txt"))

    for txt_file in txt_files:
        valid_lines = []
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if not parts: continue
                # Force class to 0 (person) - assuming LLVIP only contains pedestrians
                parts[0] = "0"
                valid_lines.append(" ".join(parts))

        if valid_lines:
            with open(txt_file, 'w') as f:
                f.write("\n".join(valid_lines) + "\n")
        else:
            os.remove(txt_file)
            img_file = txt_file.replace('/labels/', '/images/').replace('.txt', '.jpg')
            if os.path.exists(img_file): os.remove(img_file)

normalize_llvip(f"{BASE}/datasets/llvip/train/labels")
normalize_llvip(f"{BASE}/datasets/llvip/valid/labels")
print("LLVIP labels normalized to class 0 (person).")

print("Merging LLVIP into Master...")
llvip_train = merge_dataset("llvip", "train")
llvip_valid = merge_dataset("llvip", "valid")

print(f"Added {llvip_train} training and {llvip_valid} validation frames from LLVIP.")

import os

BASE = "/content/drive/MyDrive/IBVAP"
os.makedirs(f"{BASE}/datasets/ibvap_master", exist_ok=True)

yaml_content = f"""
path: {BASE}/datasets/ibvap_master
train: train/images
val: valid/images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""

yaml_path = f"{BASE}/datasets/ibvap_master/data.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

if os.path.exists(yaml_path):
    print(f"Success! {yaml_path} created.")
else:
    print(f"Error: Failed to create {yaml_path}")

model = YOLO("yolov8n.pt") # Starting fresh to prevent catastrophic forgetting

results = model.train(
    data=f"{BASE}/datasets/ibvap_master/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    workers=2,
    project=f"{BASE}/outputs",
    name="ibvap_final_detector"
)

# Backup the final model
!cp {BASE}/outputs/ibvap_final_detector/weights/best.pt {BASE}/models/ibvap_detector.pt
print("Final IBVAP model successfully backed up to Drive!")

!cp {BASE}/outputs/ibvap_final_detector-2/weights/best.pt {BASE}/models/ibvap_detector.pt
print("Final IBVAP model successfully backed up to Drive!")

# Change this line:
# detector = YOLO(f"{BASE}/models/ibvap_exdark_v1.pt")

# To this:
detector = YOLO(f"{BASE}/models/ibvap_detector.pt")

import time
import pandas as pd
import json
import cv2

## 6. Benchmarking, video playback & geometry scenarios

In [ ]:
# Download a vehicle-heavy clip to test ANPR and trajectory logic
!wget -q -O {BASE}/videos/vehicles.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4

test_videos = {
    "day_pedestrians": f"{BASE}/videos/day.mp4",
    "night_synthetic": f"{BASE}/videos/night_synthetic.mp4",
    "day_vehicles": f"{BASE}/videos/vehicles.mp4"
}

summary = []

print("Running final benchmarks (this will take a minute)...\n")

for label, path in test_videos.items():
    start = time.time()

    # Run the updated master pipeline
    run_ibvap(path, f"{BASE}/showcase/out_{label}.mp4")
    elapsed = time.time() - start

    # Load the generated alerts for this specific video
    with open(f"{BASE}/outputs/alerts.json", "r") as f:
        events = json.load(f)

    # Tally event types
    counts = {}
    for e in events:
        counts[e["event_type"]] = counts.get(e["event_type"], 0) + 1

    cap = cv2.VideoCapture(path)
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    summary_data = {
        "Scenario": label,
        "Total Frames": n_frames,
        "Process Time (s)": round(elapsed, 1),
        "Effective FPS": round(n_frames / elapsed, 2),
        **counts
    }
    summary.append(summary_data)

# Display the final benchmark table
benchmark_df = pd.DataFrame(summary).fillna(0)
display(benchmark_df)

# Save to drive for your presentation
benchmark_df.to_csv(f"{BASE}/metrics/final_results.csv", index=False)
print(f"\nMetrics saved to {BASE}/metrics/final_results.csv")

import pandas as pd
import json
import base64
import os
from IPython.display import HTML, display

# Select which processed video to view (e.g., day, night, or vehicles)
source_video = f"{BASE}/showcase/out_day_vehicles.mp4" # Update filename if needed
browser_video = "browser_ready_demo.mp4"

print(f"Re-encoding {source_video} for browser playback...")
# OpenCV uses a codec that browsers hate. ffmpeg converts it to standard H.264.
!ffmpeg -y -hide_banner -loglevel error -i {source_video} -vcodec libx264 {browser_video}

# 1. Display the Event Log
print("\n--- 📊 IBVAP Audit Log ---")
try:
    df = pd.read_json(f"{BASE}/outputs/alerts.json")
    if not df.empty:
        # Reorder for scannability
        cols = ["timestamp", "frame", "event_type", "object_type", "track_id", "severity"]
        display(df[cols].tail(10))
    else:
        print("No events triggered in this clip.")
except ValueError:
    print("No events logged.")

# 2. Display the Video Player
print("\n--- 📹 Processed Camera Feed ---")
if os.path.exists(browser_video):
    with open(browser_video, "rb") as f:
        video_encoded = base64.b64encode(f.read()).decode('ascii')

    video_html = f'''
    <video width="800" controls autoplay muted loop>
        <source src="data:video/mp4;base64,{video_encoded}" type="video/mp4">
    </video>
    '''
    display(HTML(video_html))
else:
    print("Video conversion failed.")

# Download diverse CCTV clips (Perimeter, Traffic, Night, Crowded Area)
!wget -q -O {BASE}/videos/crowd_cctv.mp4 "https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4"
!wget -q -O {BASE}/videos/traffic_cctv.mp4 "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
!wget -q -O {BASE}/videos/patrol_cctv.mp4 "https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4"

# Real low-light street CCTV sample
!wget -q -O {BASE}/videos/night_real.mp4 "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"

print("CCTV feeds downloaded successfully.")

import base64
import os
import pandas as pd
from IPython.display import HTML, display

# Select target video: "crowd_cctv.mp4", "traffic_cctv.mp4", "patrol_cctv.mp4", or "night_real.mp4"
SELECTED_CLIP = "crowd_cctv.mp4"

input_clip = f"{BASE}/videos/{SELECTED_CLIP}"
raw_output = f"{BASE}/showcase/processed_{SELECTED_CLIP}"
browser_output = f"{BASE}/showcase/browser_{SELECTED_CLIP}.mp4"

print(f"Processing feed: {SELECTED_CLIP}...")
run_ibvap(input_clip, raw_output)

# Re-encode to H.264 for inline browser playback
!ffmpeg -y -hide_banner -loglevel error -i {raw_output} -vcodec libx264 -pix_fmt yuv420p {browser_output}

# Display Event Log Summary
print("\n--- 🚨 Triggered Security Events ---")
try:
    df = pd.read_json(f"{BASE}/outputs/alerts.json")
    if not df.empty:
        cols = ["timestamp", "frame", "event_type", "object_type", "track_id", "severity"]
        display(df[[c for c in cols if c in df.columns]].tail(15))
    else:
        print("No breaches or loitering events triggered on this clip.")
except Exception as e:
    print("No events logged.")

# Render Visualized Output
print("\n--- 📹 Visual Surveillance Feed ---")
if os.path.exists(browser_output):
    with open(browser_output, "rb") as f:
        video_bytes = f.read()
    video_b64 = base64.b64encode(video_bytes).decode('ascii')

    display(HTML(f'''
    <video width="750" controls autoplay muted loop>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''))
else:
    print("Video encoding failed.")

import cv2
import json
import numpy as np
import base64
import os
import pandas as pd
from collections import deque
from datetime import datetime
from IPython.display import HTML, display
from ultralytics import YOLO

# 1. Load the sanitized final detector
detector = YOLO(f"{BASE}/models/ibvap_detector.pt")

def get_preset_geometries(width, height, layout_mode="diagonal_split"):
    """
    Presets to test varying boundary and restricted zone configurations.
    """
    if layout_mode == "diagonal_split":
        # A diagonal perimeter line crossing from top-left to bottom-right
        fence_line = ((0, int(height * 0.25)), (width, int(height * 0.85)))
        # Trapezoid restricted zone on the right flank
        zone_polygon = [
            (int(width * 0.55), int(height * 0.15)),
            (int(width * 0.95), int(height * 0.25)),
            (int(width * 0.90), int(height * 0.85)),
            (int(width * 0.45), int(height * 0.70))
        ]
    elif layout_mode == "center_quadrant":
        # Horizontal checkpoint fence across the mid-section
        fence_line = ((0, int(height * 0.55)), (width, int(height * 0.55)))
        # Central square high-security zone
        zone_polygon = [
            (int(width * 0.30), int(height * 0.30)),
            (int(width * 0.70), int(height * 0.30)),
            (int(width * 0.70), int(height * 0.75)),
            (int(width * 0.30), int(height * 0.75))
        ]
    elif layout_mode == "perimeter_corridor":
        # Vertical fence line cutting left third
        fence_line = ((int(width * 0.40), 0), (int(width * 0.40), height))
        # Deep corridor zone on the lower left
        zone_polygon = [
            (int(width * 0.05), int(height * 0.50)),
            (int(width * 0.40), int(height * 0.50)),
            (int(width * 0.40), int(height * 0.95)),
            (int(width * 0.05), int(height * 0.95))
        ]
    return fence_line, zone_polygon

def run_geometry_test(video_name, layout_mode="diagonal_split", dwell_frames=20):
    input_path = f"{BASE}/videos/{video_name}"
    raw_output = f"{BASE}/showcase/test_{layout_mode}_{video_name}"
    browser_output = f"{BASE}/showcase/browser_{layout_mode}_{video_name}.mp4"

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"Error opening video: {input_path}")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25

    out = cv2.VideoWriter(raw_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Initialize geometry based on selected preset
    fence_p, zone_poly = get_preset_geometries(width, height, layout_mode)
    fence = VirtualFence(line_p1=fence_p[0], line_p2=fence_p[1])
    restricted_zone = RestrictedZone(polygon=zone_poly, dwell_threshold_frames=dwell_frames, speed_threshold_px=28.0)
    vehicle_tracker = VehicleTracker(history_len=30)

    target_vehicles = {'car', 'bus', 'truck', 'motorcycle', 'bicycle'}
    vehicle_classes = {k for k, v in detector.names.items() if v.lower() in target_vehicles}

    event_log = []
    frame_idx = 0
    plate_cache = set()

    print(f"Running scenario [{layout_mode}] on {video_name}...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # 1. Environment: CLAHE Enhancement for dark scenes
        is_night = np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)) < 85
        if is_night:
            frame = apply_clahe(frame)

        # 2. Tracking with ByteTrack
        results = detector.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=0.30)[0]

        if results.boxes and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.int().cpu().numpy()
            classes = results.boxes.cls.int().cpu().numpy()

            for box, tid, cls_id in zip(boxes, track_ids, classes):
                cls_name = detector.names[cls_id]
                x1, y1, x2, y2 = map(int, box)
                box_color = (0, 255, 0)
                status_text = f"ID:{tid} {cls_name}"

                base_event = {
                    "timestamp": datetime.now().strftime("%H:%M:%S.%f")[:-3],
                    "frame": frame_idx,
                    "object_type": cls_name,
                    "track_id": int(tid)
                }

                # Check Perimeter Fence Line
                if fence.update(tid, box):
                    box_color = (0, 0, 255)
                    event_log.append({**base_event, "event_type": "PERIMETER_BREACH", "severity": "CRITICAL"})

                # Check Restricted Zone
                for za in restricted_zone.update(tid, box, frame_idx):
                    event_log.append({**base_event, "event_type": za["type"], "severity": "WARNING"})

                # Vehicle Trajectory & Motion
                if cls_id in vehicle_classes:
                    direction, v_events = vehicle_tracker.update_trajectory(tid, box, frame_idx)
                    status_text += f" [{direction}]"
                    for ve in v_events:
                        event_log.append({**base_event, "event_type": ve["type"], "severity": "CRITICAL"})

                    # ANPR sub-check on vehicle
                    plate_res = plate_detector(frame, conf=0.35, verbose=False)[0]
                    if plate_res.boxes:
                        for p_box in plate_res.boxes.xyxy.cpu().numpy():
                            p_text, p_conf = extract_plate_text(frame, p_box)
                            if p_text and p_text not in plate_cache:
                                plate_cache.add(p_text)
                                event_log.append({**base_event, "event_type": "ANPR_DETECT", "plate": p_text, "severity": "INFO"})

                if tid in fence.breaches:
                    box_color = (0, 0, 255)

                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(frame, status_text, (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, box_color, 2)

        # 3. Privacy Redaction
        frame = redact_faces(frame)

        # 4. Geometry HUD Overlays
        # Fence Line (Red)
        cv2.line(frame, fence.line_p1, fence.line_p2, (0, 0, 255), 3)
        cv2.putText(frame, "PERIMETER FENCE", (fence.line_p1[0] + 10, fence.line_p1[1] - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        # Semi-transparent overlay for Restricted Zone
        overlay = frame.copy()
        cv2.fillPoly(overlay, [restricted_zone.polygon], (0, 255, 255))
        cv2.addWeighted(overlay, 0.20, frame, 0.80, 0, frame)
        cv2.polylines(frame, [restricted_zone.polygon], True, (0, 255, 255), 2)
        cv2.putText(frame, f"RESTRICTED ZONE (DWELL > {dwell_frames}F)",
                    (restricted_zone.polygon[0][0], restricted_zone.polygon[0][1] - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

        out.write(frame)

    cap.release()
    out.release()

    # Save log
    with open(f"{BASE}/outputs/alerts_{layout_mode}_{video_name}.json", "w") as f:
        json.dump(event_log, f, indent=2)

    # Convert to browser playable MP4
    !ffmpeg -y -hide_banner -loglevel error -i {raw_output} -vcodec libx264 -pix_fmt yuv420p {browser_output}

    # Render results
    print(f"\n--- 🚨 Security Event Log ({len(event_log)} total events) ---")
    if event_log:
        df = pd.DataFrame(event_log)
        display(df.tail(12))
    else:
        print("No trigger events recorded in this configuration.")

    print("\n--- 📹 Visual Surveillance Feed ---")
    with open(browser_output, "rb") as f:
        video_b64 = base64.b64encode(f.read()).decode('ascii')
    display(HTML(f'''
    <video width="750" controls autoplay muted loop>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
    </video>
    '''))

run_geometry_test("crowd_cctv.mp4", layout_mode="diagonal_split", dwell_frames=20)

run_geometry_test("traffic_cctv.mp4", layout_mode="center_quadrant", dwell_frames=15)

run_geometry_test("night_real.mp4", layout_mode="perimeter_corridor", dwell_frames=10)

## 7. License-plate model training & camera configuration

In [ ]:
BASE = "/content/drive/MyDrive/IBVAP"
os.makedirs(f"{BASE}/datasets/plates", exist_ok=True)
os.makedirs(f"{BASE}/models", exist_ok=True)
os.makedirs(f"{BASE}/outputs", exist_ok=True)
os.makedirs(f"{BASE}/metrics", exist_ok=True)

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(13)
dataset = version.download("yolov8")

LATEST_VERSION = 13
version = project.version(LATEST_VERSION)
dataset = version.download("yolov8")

PLATES_DIR = f"{BASE}/datasets/plates"

# Skip copying dataset to Drive — train directly from local Colab storage
PLATES_DIR = dataset.location  # e.g. /content/license-plate-recognition-rxg4e-13

yaml_path = f"{PLATES_DIR}/data.yaml"
print("\n========== data.yaml ==========\n")
with open(yaml_path, "r") as f:
    print(f.read())

from collections import Counter

label_dir = f"{PLATES_DIR}/train/labels"
counter = Counter()
for file in os.listdir(label_dir):
    if file.endswith(".txt"):
        with open(os.path.join(label_dir, file)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1

print("Class distribution:", counter)
print(f"Training images: {len(os.listdir(f'{PLATES_DIR}/train/images'))}")

plate_model = YOLO("yolov8n.pt")

results = plate_model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    workers=2,
    project=f"{BASE}/outputs",
    name="plate_yolov8n"
)

import numpy as np
import cv2
from collections import deque

def ccw(A, B, C):
    return (C[1] - A[1]) * (B[0] - A[0]) > (B[1] - A[1]) * (C[0] - A[0])

def intersect(A, B, C, D):
    return ccw(A, C, D) != ccw(B, C, D) and ccw(A, B, C) != ccw(A, B, D)

class VirtualFence:
    def __init__(self, line_p1, line_p2):
        self.line_p1 = tuple(line_p1)
        self.line_p2 = tuple(line_p2)
        self.track_history = {}
        self.breaches = set()

    def update(self, track_id, bbox):
        x1, y1, x2, y2 = bbox
        bottom_center = ((x1 + x2) / 2, y2)
        if track_id not in self.track_history:
            self.track_history[track_id] = []
        self.track_history[track_id].append(bottom_center)

        if len(self.track_history[track_id]) >= 2:
            prev_pt = self.track_history[track_id][-2]
            curr_pt = self.track_history[track_id][-1]
            if intersect(prev_pt, curr_pt, self.line_p1, self.line_p2):
                if track_id not in self.breaches:
                    self.breaches.add(track_id)
                    return True
        return False

class RestrictedZone:
    def __init__(self, polygon, dwell_threshold_frames=30, speed_threshold_px=35.0, smooth_window=3):
        self.polygon = np.array(polygon, np.int32)
        self.dwell_threshold = dwell_threshold_frames
        self.speed_threshold = speed_threshold_px
        self.smooth_window = smooth_window
        self.entry_frames = {}
        self.position_history = {}
        self.alerted_loitering = set()

    def update(self, track_id, bbox, frame_idx):
        x1, y1, x2, y2 = bbox
        cx, cy = int((x1 + x2) / 2), int(y2)
        is_inside = cv2.pointPolygonTest(self.polygon, (cx, cy), False) >= 0
        alerts = []

        if track_id not in self.position_history:
            self.position_history[track_id] = deque(maxlen=self.smooth_window)
        self.position_history[track_id].append((cx, cy))

        if is_inside:
            if track_id not in self.entry_frames:
                self.entry_frames[track_id] = frame_idx
            dwell = frame_idx - self.entry_frames[track_id]
            if dwell >= self.dwell_threshold and track_id not in self.alerted_loitering:
                self.alerted_loitering.add(track_id)
                alerts.append({"type": "LOITERING_ALERT", "track_id": int(track_id), "dwell_frames": int(dwell)})
        else:
            self.entry_frames.pop(track_id, None)
            self.alerted_loitering.discard(track_id)

        # 3-frame rolling smoothed speed calculation
        if len(self.position_history[track_id]) >= self.smooth_window:
            pts = list(self.position_history[track_id])
            displacements = [
                np.sqrt((pts[i][0] - pts[i-1][0])**2 + (pts[i][1] - pts[i-1][1])**2)
                for i in range(1, len(pts))
            ]
            smoothed_speed = float(np.mean(displacements))
            if smoothed_speed > self.speed_threshold:
                alerts.append({"type": "SUDDEN_MOVEMENT", "track_id": int(track_id), "speed_px": round(smoothed_speed, 2)})

        return alerts

class VehicleTracker:
    def __init__(self, history_len=30):
        self.trajectories = {}
        self.history_len = history_len
        self.wrong_way_alerts = set()

    def update_trajectory(self, track_id, bbox, frame_idx):
        x1, y1, x2, y2 = bbox
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        if track_id not in self.trajectories:
            self.trajectories[track_id] = deque(maxlen=self.history_len)
        self.trajectories[track_id].append((cx, cy))

        direction = "STATIONARY"
        events = []
        if len(self.trajectories[track_id]) >= 10:
            dx = self.trajectories[track_id][-1][0] - self.trajectories[track_id][0][0]
            dy = self.trajectories[track_id][-1][1] - self.trajectories[track_id][0][1]
            direction = ("RIGHT" if dx > 0 else "LEFT") if abs(dx) > abs(dy) else ("DOWN" if dy > 0 else "UP")

            if direction == "DOWN" and track_id not in self.wrong_way_alerts:
                self.wrong_way_alerts.add(track_id)
                events.append({"frame": frame_idx, "type": "WRONG_WAY_VEHICLE", "track_id": int(track_id), "direction": direction})

        return direction, events

import json
import os

BASE = "/content/drive/MyDrive/IBVAP"
config_path = f"{BASE}/cameras.json"

cameras_config = {
    "CAM-BOP-01": {
        "description": "Main Approach Gate - Perimeter",
        "fence_line": [[0, 0.70], [1.0, 0.70]],  # Normalized (x, y) coordinates
        "restricted_zone": [
            [0.25, 0.35],
            [0.75, 0.35],
            [0.75, 0.85],
            [0.25, 0.85]
        ],
        "dwell_threshold_frames": 30,
        "speed_threshold_px": 32.0,
        "motion_min_area": 1200
    },
    "CAM-PERIMETER-02": {
        "description": "Flank Checkpoint - Diagonal Cut",
        "fence_line": [[0.0, 0.25], [1.0, 0.85]],
        "restricted_zone": [
            [0.55, 0.15],
            [0.95, 0.25],
            [0.90, 0.85],
            [0.45, 0.70]
        ],
        "dwell_threshold_frames": 20,
        "speed_threshold_px": 28.0,
        "motion_min_area": 1500
    }
}

with open(config_path, "w") as f:
    json.dump(cameras_config, f, indent=2)

print(f"Configurations saved to {config_path}")

import cv2
import json
import numpy as np
from datetime import datetime
from ultralytics import YOLO

BASE = "/content/drive/MyDrive/IBVAP"

# Verify models
detector = YOLO(f"{BASE}/models/ibvap_detector.pt")
plate_model_path = f"{BASE}/models/license_plate_detector.pt"
plate_detector = YOLO(plate_model_path)

## 8. Optimized production pipeline & showcase generation

In [ ]:
def run_ibvap_optimized(input_path, output_path, camera_id="CAM-BOP-01", config_file=f"{BASE}/cameras.json"):
    with open(config_file, "r") as f:
        cfg = json.load(f)[camera_id]

    cap = cv2.VideoCapture(input_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Scale normalized coordinates from config to frame dimensions
    p1 = (int(cfg["fence_line"][0][0] * width), int(cfg["fence_line"][0][1] * height))
    p2 = (int(cfg["fence_line"][1][0] * width), int(cfg["fence_line"][1][1] * height))
    zone_pts = [(int(pt[0] * width), int(pt[1] * height)) for pt in cfg["restricted_zone"]]

    fence = VirtualFence(line_p1=p1, line_p2=p2)
    zone = RestrictedZone(
        polygon=zone_pts,
        dwell_threshold_frames=cfg.get("dwell_threshold_frames", 30),
        speed_threshold_px=cfg.get("speed_threshold_px", 35.0),
        smooth_window=3
    )
    tracker = VehicleTracker()

    # MOG2 Cascade Gate Initialization
    fgbg = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=32, detectShadows=False)
    min_motion_area = cfg.get("motion_min_area", 1200)

    vehicle_classes = {'car', 'bus', 'truck', 'motorcycle', 'bicycle'}
    vehicle_cls_ids = {k for k, v in detector.names.items() if v.lower() in vehicle_classes}

    event_log = []
    frame_idx = 0
    plate_cache = set()
    skipped_frames = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        # 1. Night Enhancement Stage
        is_night = np.mean(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)) < 85
        if is_night:
            frame = apply_clahe(frame)

        # 2. MOG2 Motion Gating Check
        fgmask = fgbg.apply(frame)
        motion_pixels = cv2.countNonZero(fgmask)

        # If zero dynamic activity, bypass heavy inference to conserve edge GPU cycles
        if motion_pixels < min_motion_area:
            skipped_frames += 1
            # Render HUD and write frame directly
            cv2.putText(frame, "MOTION GATE: IDLE / LOW DYNAMICS", (10, height - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)
            cv2.line(frame, fence.line_p1, fence.line_p2, (0, 0, 255), 2)
            cv2.polylines(frame, [zone.polygon], True, (255, 255, 0), 2)
            out.write(frame)
            continue

        # 3. Primary Object Track Engine
        results = detector.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=0.30)[0]
        has_vehicles = False

        if results.boxes and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            track_ids = results.boxes.id.int().cpu().numpy()
            classes = results.boxes.cls.int().cpu().numpy()

            for box, tid, cls_id in zip(boxes, track_ids, classes):
                cls_name = detector.names[cls_id]
                x1, y1, x2, y2 = map(int, box)
                box_color = (0, 255, 0)
                status_label = f"ID:{tid} {cls_name}"

                base_event = {
                    "camera_id": camera_id,
                    "timestamp": datetime.now().isoformat(),
                    "frame": frame_idx,
                    "object_type": cls_name,
                    "track_id": int(tid)
                }

                # Perimeter breach check
                if fence.update(tid, box):
                    box_color = (0, 0, 255)
                    event_log.append({**base_event, "event_type": "PERIMETER_BREACH", "severity": "CRITICAL"})

                # Restricted zone checks (Loitering & Smoothed Speed)
                for alert in zone.update(tid, box, frame_idx):
                    event_log.append({**base_event, "event_type": alert["type"], "severity": "WARNING"})

                # Vehicle trajectory logic
                if cls_id in vehicle_cls_ids:
                    has_vehicles = True
                    direction, v_events = tracker.update_trajectory(tid, box, frame_idx)
                    status_label += f" [{direction}]"
                    for ve in v_events:
                        event_log.append({**base_event, "event_type": ve["type"], "severity": "CRITICAL"})

                if tid in fence.breaches:
                    box_color = (0, 0, 255)

                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
                cv2.putText(frame, status_label, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, box_color, 2)

        # 4. Decoupled ANPR: Run once per frame ONLY if vehicles exist in scene
        if has_vehicles:
            plate_res = plate_detector(frame, conf=0.35, verbose=False)[0]
            if plate_res.boxes:
                for p_box in plate_res.boxes.xyxy.cpu().numpy():
                    text, conf = extract_plate_text(frame, p_box)
                    if text and text not in plate_cache:
                        plate_cache.add(text)
                        event_log.append({
                            "camera_id": camera_id,
                            "timestamp": datetime.now().isoformat(),
                            "frame": frame_idx,
                            "object_type": "license_plate",
                            "track_id": -1,
                            "event_type": "ANPR_HIT",
                            "plate": text,
                            "confidence": round(conf, 2),
                            "severity": "INFO"
                        })

        # 5. Face Obfuscation & HUD Overlays
        frame = redact_faces(frame)
        cv2.line(frame, fence.line_p1, fence.line_p2, (0, 0, 255), 2)
        cv2.polylines(frame, [zone.polygon], True, (0, 255, 255), 2)
        mode_text = f"{camera_id} | NIGHT ACTIVE" if is_night else f"{camera_id} | DAY ACTIVE"
        cv2.putText(frame, mode_text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()

    alerts_path = f"{BASE}/outputs/alerts_{camera_id}.json"
    with open(alerts_path, "w") as f:
        json.dump(event_log, f, indent=2)

    print(f"[{camera_id}] Processed {frame_idx} frames. MOG2 gated {skipped_frames} idle frames.")
    return alerts_path

import cv2
import numpy as np

# Process Day and Synthetic Night inputs
print("Generating Day clip...")
run_ibvap_optimized(f"{BASE}/videos/day.mp4", f"{BASE}/showcase/day_processed.mp4", camera_id="CAM-BOP-01")

print("Generating Night clip...")
run_ibvap_optimized(f"{BASE}/videos/night_synthetic.mp4", f"{BASE}/showcase/night_processed.mp4", camera_id="CAM-BOP-01")

def build_side_by_side_showcase(day_path, night_path, output_path):
    cap_day = cv2.VideoCapture(day_path)
    cap_night = cv2.VideoCapture(night_path)

    w = int(cap_day.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap_day.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap_day.get(cv2.CAP_PROP_FPS)) or 25

    # Target: Double width side-by-side with bottom telemetry bar
    banner_height = 80
    combined_width = w * 2
    combined_height = h + banner_height

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (combined_width, combined_height))

    while True:
        ret_d, f_day = cap_day.read()
        ret_n, f_night = cap_night.read()

        if not ret_d or not ret_n:
            break

        # Resize safety match
        if (f_night.shape[1], f_night.shape[0]) != (w, h):
            f_night = cv2.resize(f_night, (w, h))

        # Horizontal stacking
        stacked = np.hstack([f_day, f_night])

        # Bottom Telemetry Canvas
        canvas = np.zeros((combined_height, combined_width, 3), dtype=np.uint8)
        canvas[0:h, 0:combined_width] = stacked

        # Header Titles
        cv2.putText(canvas, "DAY OPERATIONAL FEED", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(canvas, "LOW-LIGHT CLAHE + EXDARK", (w + 30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        # Dashboard / Audit metrics banner
        cv2.line(canvas, (0, h), (combined_width, h), (70, 70, 70), 2)
        cv2.putText(canvas, "IBVAP EDGE SURVEILLANCE PIPELINE  |  MOG2 CASCADE ACTIVE  |  BYTETRACK FUSED",
                    (30, h + 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)
        cv2.putText(canvas, "ANALYTICS: BREACH DETECTION | LOITERING DWELL | ANPR EXTRACTION | PRIVACY MASKING",
                    (30, h + 62), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        out.write(canvas)

    cap_day.release()
    cap_night.release()
    out.release()
    print("Stitching complete.")

raw_showcase = f"{BASE}/showcase/ibvap_showcase_raw.mp4"
final_h264 = f"{BASE}/showcase/ibvap_side_by_side_demo.mp4"

build_side_by_side_showcase(
    f"{BASE}/showcase/day_processed.mp4",
    f"{BASE}/showcase/night_processed.mp4",
    raw_showcase
)

# Convert to H.264
!ffmpeg -y -hide_banner -loglevel error -i {raw_showcase} -vcodec libx264 -pix_fmt yuv420p {final_h264}
print(f"Ready for presentation: {final_h264}")

# 1. Swap the trained model into production once training completes
plate_backup_path = f"{BASE}/outputs/plate_yolov8n/weights/best.pt"
target_prod_path = f"{BASE}/models/license_plate_detector.pt"

if os.path.exists(plate_backup_path):
    !cp {plate_backup_path} {target_prod_path}
    print("Successfully replaced generic plate detector with fine-tuned Indian Plate weights.")
    # Reload model directly
    plate_detector = YOLO(target_prod_path)
else:
    print(f"Weights not found at {plate_backup_path}. Verify that training completed.")

import base64
import os
from IPython.display import HTML, display

BASE = "/content/drive/MyDrive/IBVAP"
demo_video_path = f"{BASE}/showcase/ibvap_side_by_side_demo.mp4"

if os.path.exists(demo_video_path):
    with open(demo_video_path, "rb") as f:
        video_bytes = f.read()
    video_b64 = base64.b64encode(video_bytes).decode('ascii')

    display(HTML(f'''
    <div style="display: flex; flex-direction: column; align-items: center;">
        <h3 style="color: #4CAF50; font-family: monospace;">IBVAP DUAL-STREAM DEMO (DAY vs CLAHE+EXDARK)</h3>
        <video width="900" controls autoplay muted loop style="border: 2px solid #333; border-radius: 8px;">
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
    </div>
    '''))
else:
    print(f"File not found: {demo_video_path}. Verify that Step 5 completed successfully.")

import json
import glob
import os
import pandas as pd
from IPython.display import display

BASE = "/content/drive/MyDrive/IBVAP"
alert_files = glob.glob(f"{BASE}/outputs/alerts_*.json")

aggregated_records = []

for file in alert_files:
    cam_name = os.path.basename(file).replace("alerts_", "").replace(".json", "")
    with open(file, "r") as f:
        try:
            records = json.load(f)
            for r in records:
                aggregated_records.append(r)
        except Exception:
            continue

if aggregated_records:
    df_all = pd.DataFrame(aggregated_records)

    # 1. Total event distribution breakdown
    print("================ 📊 SYSTEM METRICS SUMMARY ================")
    summary_pivot = pd.crosstab(
        df_all["camera_id"],
        df_all["event_type"],
        margins=True,
        margins_name="Total"
    )
    display(summary_pivot)

    # 2. Preview recent critical alerts
    print("\n================ 🚨 RECENT HIGH-SEVERITY ALERTS ================")
    crit_cols = ["timestamp", "camera_id", "frame", "event_type", "object_type", "severity"]
    crit_df = df_all[df_all["severity"].isin(["CRITICAL", "WARNING"])].tail(10)
    display(crit_df[[c for c in crit_cols if c in crit_df.columns]])

    # Export to CSV for presentation backup
    summary_pivot.to_csv(f"{BASE}/metrics/event_summary_table.csv")
    print(f"\nSaved dashboard metrics to {BASE}/metrics/event_summary_table.csv")
else:
    print("No events found across alert logs. Ensure videos have completed inference.")

import cv2
import matplotlib.pyplot as plt

# Test plate model on a frame from the vehicle test video
cap = cv2.VideoCapture(f"{BASE}/videos/day_vehicles.mp4" if os.path.exists(f"{BASE}/videos/day_vehicles.mp4") else f"{BASE}/videos/day.mp4")
ret, sample_frame = cap.read()
cap.release()

if ret:
    plate_res = plate_detector(sample_frame, conf=0.25, verbose=False)[0]
    annotated = sample_frame.copy()

    if plate_res.boxes:
        for box in plate_res.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            text, conf = extract_plate_text(sample_frame, box)
            label = f"{text} ({conf:.2f})" if text else "Plate"
            cv2.putText(annotated, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 6))
    plt.imshow(annotated_rgb)
    plt.axis("off")
    plt.title("Indian ANPR Sanity Check (YOLOv8 + EasyOCR)")
    plt.show()
else:
    print("Unable to extract test frame.")

import time
import cv2
import pandas as pd
import numpy as np

test_clip = f"{BASE}/videos/day.mp4"
sample_frames = 150

def benchmark_run(use_mog2=True):
    cap = cv2.VideoCapture(test_clip)
    fgbg = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=32, detectShadows=False) if use_mog2 else None

    detector_calls = 0
    start = time.time()
    count = 0

    while cap.isOpened() and count < sample_frames:
        ret, frame = cap.read()
        if not ret:
            break
        count += 1

        if use_mog2:
            fgmask = fgbg.apply(frame)
            if cv2.countNonZero(fgmask) < 1200:
                continue

        # Inference executed
        _ = detector(frame, verbose=False, conf=0.30)
        detector_calls += 1

    cap.release()
    elapsed = time.time() - start
    return count, detector_calls, elapsed

print("Profiling Pipeline Efficiency...")
total_f, gated_calls, time_mog2 = benchmark_run(use_mog2=True)
_, raw_calls, time_raw = benchmark_run(use_mog2=False)

reduction_pct = ((raw_calls - gated_calls) / raw_calls) * 100
speedup_pct = ((time_raw - time_mog2) / time_raw) * 100

profile_data = {
    "Metric": [
        "Total Evaluated Frames",
        "Deep Model Forward Passes",
        "Frames Gated (Skipped)",
        "Compute Reduction",
        "Total Latency",
        "Effective Throughput",
        "Edge Power/Latency Savings"
    ],
    "Baseline (Always Infer)": [
        total_f,
        raw_calls,
        0,
        "0.0%",
        f"{time_raw:.2f} s",
        f"{total_f / time_raw:.1f} FPS",
        "Baseline"
    ],
    "IBVAP (MOG2 Cascade Gate)": [
        total_f,
        gated_calls,
        total_f - gated_calls,
        f"{reduction_pct:.1f}%",
        f"{time_mog2:.2f} s",
        f"{total_f / time_mog2:.1f} FPS",
        f"{speedup_pct:.1f}% faster"
    ]
}

profile_df = pd.DataFrame(profile_data)
display(profile_df)
profile_df.to_csv(f"{BASE}/metrics/mog2_efficiency_profile.csv", index=False)
print(f"\n✓ Profiling table exported to {BASE}/metrics/mog2_efficiency_profile.csv")

import shutil
import os
from google.colab import files

export_dir = "/content/IBVAP_Pitch_Assets"
os.makedirs(export_dir, exist_ok=True)

# Collect essential pitch assets
artifacts = [
    (f"{BASE}/showcase/ibvap_side_by_side_demo.mp4", "ibvap_side_by_side_demo.mp4"),
    (f"{BASE}/cameras.json", "cameras.json"),
    (f"{BASE}/metrics/event_summary_table.csv", "event_summary_table.csv"),
    (f"{BASE}/metrics/mog2_efficiency_profile.csv", "mog2_efficiency_profile.csv"),
    (f"{BASE}/outputs/alerts_CAM-BOP-01.json", "alerts_sample.json")
]

copied = 0
for src, name in artifacts:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(export_dir, name))
        copied += 1
    else:
        print(f"Skipped missing asset: {src}")

zip_path = "/content/IBVAP_Pitch_Assets"
shutil.make_archive(zip_path, 'zip', export_dir)
print(f"✓ Packaged {copied} core assets into /content/IBVAP_Pitch_Assets.zip")

# Trigger automatic browser download
files.download(f"{zip_path}.zip")

## 9. Multi-terrain dataset expansion & validation

In [ ]:
import os
from roboflow import Roboflow

BASE = "/content/drive/MyDrive/IBVAP"
TERRAIN_DIR = f"{BASE}/datasets/terrain"
os.makedirs(TERRAIN_DIR, exist_ok=True)

# Using Roboflow Border/Terrain Surveillance Project
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("surveillance-datasets").project("border-and-terrain-detection")
version = project.version(1)
dataset = version.download("yolov8")

# If local path is used, point to it
TERRAIN_RAW_DIR = dataset.location
print(f"Downloaded terrain data to: {TERRAIN_RAW_DIR}")

import os
import glob
import yaml

# Read incoming data.yaml to see source class mappings
raw_yaml_path = f"{TERRAIN_RAW_DIR}/data.yaml"
with open(raw_yaml_path, "r") as f:
    raw_yaml = yaml.safe_load(f)

print("Incoming dataset classes:", raw_yaml.get("names"))

# Setup explicit dictionary mapping from incoming names to IBVAP master indices
# Adjust the keys here based on the exact printed names from raw_yaml
TARGET_MASTER_MAP = {
    "person": 0,
    "pedestrian": 0,
    "soldier": 0,
    "human": 0,
    "bicycle": 1,
    "bike": 1,
    "car": 2,
    "vehicle": 2,
    "motorcycle": 3,
    "motorbike": 3,
    "bus": 4,
    "truck": 5
}

def sanitize_and_remap_terrain(split="train"):
    lbl_dir = os.path.join(TERRAIN_RAW_DIR, split, "labels")
    img_dir = os.path.join(TERRAIN_RAW_DIR, split, "images")
    if not os.path.exists(lbl_dir):
        return

    src_names = raw_yaml.get("names", {})
    if isinstance(src_names, list):
        src_names = {i: name for i, name in enumerate(src_names)}

    files = glob.glob(os.path.join(lbl_dir, "*.txt"))
    kept_count, dropped_count = 0, 0

    for txt_path in files:
        new_lines = []
        with open(txt_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                src_cls_id = int(parts[0])
                src_name = str(src_names.get(src_cls_id, "")).lower()

                if src_name in TARGET_MASTER_MAP:
                    parts[0] = str(TARGET_MASTER_MAP[src_name])
                    new_lines.append(" ".join(parts))

        # Overwrite if valid objects exist; otherwise remove frame to avoid unannotated negatives
        if new_lines:
            with open(txt_path, "w") as f:
                f.write("\n".join(new_lines) + "\n")
            kept_count += 1
        else:
            os.remove(txt_path)
            stem = os.path.splitext(os.path.basename(txt_path))[0]
            for ext in [".jpg", ".png", ".jpeg"]:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    os.remove(img_path)
            dropped_count += 1

    print(f"[{split}] Retained {kept_count} matching annotations, removed {dropped_count} unmatched/empty frames.")

sanitize_and_remap_terrain("train")
sanitize_and_remap_terrain("valid")

import os
import shutil

MASTER_DIR = f"{BASE}/datasets/ibvap_master"

def merge_terrain_to_master(split="train"):
    src_img = os.path.join(TERRAIN_RAW_DIR, split, "images")
    src_lbl = os.path.join(TERRAIN_RAW_DIR, split, "labels")

    dst_img = os.path.join(MASTER_DIR, split, "images")
    dst_lbl = os.path.join(MASTER_DIR, split, "labels")

    os.makedirs(dst_img, exist_ok=True)
    os.makedirs(dst_lbl, exist_ok=True)

    copied = 0
    for file in os.listdir(src_img):
        new_basename = f"terrain_{file}"
        shutil.copy(os.path.join(src_img, file), os.path.join(dst_img, new_basename))

        label_name = os.path.splitext(file)[0] + ".txt"
        lbl_source = os.path.join(src_lbl, label_name)
        if os.path.exists(lbl_source):
            new_lbl_name = f"terrain_{label_name}"
            shutil.copy(lbl_source, os.path.join(dst_lbl, new_lbl_name))
        copied += 1
    return copied

added_train = merge_terrain_to_master("train")
added_val = merge_terrain_to_master("valid")
print(f"Merged into master dataset: +{added_train} train images, +{added_val} valid images.")

from collections import Counter
import os

MASTER_DIR = f"{BASE}/datasets/ibvap_master"
label_dir = f"{MASTER_DIR}/train/labels"

counter = Counter()
for file in os.listdir(label_dir):
    if file.endswith(".txt"):
        with open(os.path.join(label_dir, file)) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1

class_names = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 4: "bus", 5: "truck"}
print("==== Unified Dataset Class Distribution ====")
for k in sorted(counter.keys()):
    print(f"Class {k} ({class_names.get(k, 'Unknown')}): {counter[k]} annotations")

# Ensure data.yaml points correctly to the merged dataset
yaml_content = f"""
path: {MASTER_DIR}
train: train/images
val: valid/images

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
"""
with open(f"{MASTER_DIR}/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print(f"\nUpdated {MASTER_DIR}/data.yaml confirmed.")

from ultralytics import YOLO

# Warm start from existing detector checkpoint
weights_to_load = f"{BASE}/models/ibvap_detector.pt"
model = YOLO(weights_to_load)

results = model.train(
    data=f"{BASE}/datasets/ibvap_master/data.yaml",
    epochs=35,
    imgsz=640,
    batch=16,
    patience=8,
    workers=2,
    # Terrain-optimized augmentations:
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    mixup=0.15,
    flipud=0.0,
    fliplr=0.5,
    project=f"{BASE}/outputs",
    name="ibvap_multiterrain_detector"
)

# Promote new checkpoint as the master model
!cp {BASE}/outputs/ibvap_multiterrain_detector/weights/best.pt {BASE}/models/ibvap_detector.pt
print("✓ Multi-terrain checkpoint saved to Drive as models/ibvap_detector.pt")

from ultralytics import YOLO

model = YOLO(f"{BASE}/models/ibvap_detector.pt")
metrics = model.val(data=f"{BASE}/datasets/ibvap_master/data.yaml", imgsz=640)

print("\n================ 📈 MULTI-TERRAIN VALIDATION METRICS ================")
print(f"mAP@50      : {metrics.box.map50:.4f}")
print(f"mAP@50-95   : {metrics.box.map:.4f}")
print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")

# Save metrics for slides
with open(f"{BASE}/metrics/terrain_model_metrics.txt", "w") as f:
    f.write(f"mAP50: {metrics.box.map50:.4f}\nmAP50-95: {metrics.box.map:.4f}\nPrecision: {metrics.box.mp:.4f}\nRecall: {metrics.box.mr:.4f}\n")

print(f"Saved metric breakdown to {BASE}/metrics/terrain_model_metrics.txt")

## 10. Lightweight motion-gating optimization

In [ ]:
import cv2
import numpy as np

class TinyMotionGate:
    """
    Ultra-lightweight temporal motion trigger (< 0.5ms execution).
    Operates on a heavy downscaled grid (160x120) using running average differencing.
    """
    def __init__(self, target_size=(160, 120), motion_thresh=18, min_motion_ratio=0.008, alpha=0.05):
        self.target_size = target_size
        self.motion_thresh = motion_thresh
        self.min_motion_pixels = int(target_size[0] * target_size[1] * min_motion_ratio)
        self.alpha = alpha  # Background adaptation rate
        self.bg_model = None

    def update(self, frame):
        """
        Returns:
            triggered (bool): True if dynamic movement is confirmed.
            motion_ratio (float): Percentage of the frame with motion.
            motion_bbox (tuple or None): (x1, y1, x2, y2) in original frame coordinates.
        """
        orig_h, orig_w = frame.shape[:2]

        # 1. Extreme downsampling & grayscale conversion (drastically cuts memory/CPU)
        small_gray = cv2.cvtColor(cv2.resize(frame, self.target_size, interpolation=cv2.INTER_NEAREST), cv2.COLOR_BGR2GRAY)
        small_gray = cv2.GaussianBlur(small_gray, (5, 5), 0)

        # Initialize background model
        if self.bg_model is None:
            self.bg_model = small_gray.astype("float32")
            return False, 0.0, None

        # 2. Fast running average background update: B = (1 - a)*B + a*I
        cv2.accumulateWeighted(small_gray, self.bg_model, self.alpha)

        # 3. Absolute temporal difference
        delta = cv2.absdiff(small_gray, cv2.convertScaleAbs(self.bg_model))
        _, thresh = cv2.threshold(delta, self.motion_thresh, 255, cv2.THRESH_BINARY)

        # Fast noise suppression
        thresh = cv2.dilate(thresh, None, iterations=1)
        active_pixels = cv2.countNonZero(thresh)
        motion_ratio = active_pixels / (self.target_size[0] * self.target_size[1])

        # 4. Trigger decision
        if active_pixels < self.min_motion_pixels:
            return False, motion_ratio, None

        # 5. Extract Coarse Dynamic Bounding Box to crop for YOLO
        nz = cv2.findNonZero(thresh)
        if nz is not None:
            rx, ry, rw, rh = cv2.boundingRect(nz)
            # Scale coordinates back up to original frame dimensions
            scale_x = orig_w / self.target_size[0]
            scale_y = orig_h / self.target_size[1]
            motion_bbox = (
                int(rx * scale_x),
                int(ry * scale_y),
                int((rx + rw) * scale_x),
                int((ry + rh) * scale_y)
            )
        else:
            motion_bbox = (0, 0, orig_w, orig_h)

        return True, motion_ratio, motion_bbox

import time

tiny_gate = TinyMotionGate(target_size=(160, 120))
mog2_gate = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=32, detectShadows=False)

cap = cv2.VideoCapture(f"{BASE}/videos/day.mp4")
frames = []
for _ in range(120):
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()

# 1. Benchmark TinyMotionGate
start_tiny = time.time()
for f in frames:
    _ = tiny_gate.update(f)
time_tiny = (time.time() - start_tiny) / len(frames) * 1000

# 2. Benchmark Full MOG2
start_mog2 = time.time()
for f in frames:
    mask = mog2_gate.apply(f)
    _ = cv2.countNonZero(mask)
time_mog2 = (time.time() - start_mog2) / len(frames) * 1000

print(f"=== LATENCY PER FRAME ===")
print(f"OpenCV MOG2 (Full Res) : {time_mog2:.2f} ms")
print(f"TinyMotionGate (Sub-160): {time_tiny:.2f} ms")
print(f"Speedup                : {time_mog2 / time_tiny:.1f}x faster")

tiny_gate = TinyMotionGate(target_size=(160, 120), motion_thresh=18, min_motion_ratio=0.008)

# Inside your cap.isOpened() loop:
# ----------------------------------------------------
is_moving, motion_ratio, motion_roi = tiny_gate.update(frame)

if not is_moving:
    # Frame is static — skip all YOLO, ANPR, and Tracking calls completely
    cv2.putText(frame, "STATUS: STATIC (GPU DORMANT)", (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (128, 128, 128), 2)
    out.write(frame)
    continue

# If motion is detected, trigger the detector
# Optional: run detector on the cropped motion_roi instead of the full frame for 2x faster YOLO
results = detector.track(frame, persist=True, tracker="bytetrack.yaml", verbose=False, conf=0.30)[0]
# ----------------------------------------------------

## Run order

For a fresh Colab runtime, execute the sections from top to bottom.

### Before Roboflow cells

```python
import os
os.environ["ROBOFLOW_API_KEY"] = "YOUR_KEY"
```

### Practical workflow

1. Mount Drive and create the IBVAP directory tree.
2. Download/prepare datasets.
3. Normalize labels into the master class mapping.
4. Train and validate the detector.
5. Load auxiliary ANPR/face models.
6. Run the inference pipeline on test footage.
7. Run benchmarks and geometry tests.
8. Generate showcase videos and export metrics.

The notebook is based on the supplied source file and keeps its original pipeline components while removing the exposed API credential.